# CryptoTrade Pro — AI/ML Security Audit

This notebook walks through a full AI/ML security audit of the **CryptoTrade Pro** trading platform.

You will learn how to:
- Identify ML-specific vulnerabilities
- Detect unsafe deserialization (pickle)
- Detect hardcoded secrets
- Detect command injection in data pipelines
- Scan notebooks for insecure patterns
- Use Bandit, Semgrep, Safety, and Pylint
- Generate a professional audit summary

This notebook mirrors the workflow used in the **SEC-mandated 72-hour breach audit** described in the course.

## 1. Environment Setup

In [ ]:
!pip install bandit semgrep safety pylint pandas -q
import os
os.environ['PATH'] += f":{os.path.expanduser('~')}/.local/bin"
print('Environment ready.')

## 2. Project Structure Overview

The CryptoTrade Pro project contains:

- `src/model_management.py` — vulnerable model loading
- `src/data_pipeline.py` — vulnerable data pipeline
- `rules/*.yaml` — custom Semgrep rules
- `notebooks/data_pipeline.ipynb` — vulnerable notebook
- `trades/data.csv` — sample trading data

We will scan all of these.

## 3. Bandit — Python Static Analysis

In [ ]:
!bandit -r src -f json -o security-reports/bandit.json
print('Bandit report saved.')

In [ ]:
import json
with open('security-reports/bandit.json') as f:
    bandit = json.load(f)
len(bandit.get('results', []))

In [ ]:
bandit_findings = [
    {
        'severity': r.get('issue_severity'),
        'rule': r.get('test_id'),
        'cwe': r.get('issue_cwe', {}).get('id'),
        'file': r.get('filename'),
        'line': r.get('line_number'),
        'text': r.get('issue_text')
    }
    for r in bandit.get('results', [])
]
bandit_findings[:10]

### Bandit Severity, CWE, and File Breakdown

In [ ]:
import pandas as pd
df_bandit = pd.DataFrame(bandit_findings)

print('=== Bandit Severity Breakdown ===')
print(df_bandit['severity'].value_counts(), '\n')

print('=== Bandit Findings by CWE ===')
print(df_bandit.groupby('cwe').size(), '\n')

print('=== Bandit Findings by File ===')
print(df_bandit.groupby('file').size(), '\n')

## 4. Semgrep — Custom ML Security Rules

In [ ]:
!semgrep --config rules --json --output security-reports/semgrep.json || true
print('Semgrep report saved.')

In [ ]:
with open('security-reports/semgrep.json') as f:
    semgrep = json.load(f)
len(semgrep.get('results', []))

In [ ]:
semgrep_findings = [
    {
        'rule': r.get('check_id'),
        'file': r.get('path'),
        'line': r.get('start', {}).get('line'),
        'message': r.get('extra', {}).get('message')
    }
    for r in semgrep.get('results', [])
]
semgrep_findings[:10]

### Semgrep Rule and File Breakdown

In [ ]:
df_semgrep = pd.DataFrame(semgrep_findings)

print('=== Semgrep Findings by Rule ===')
print(df_semgrep.groupby('rule').size(), '\n')

print('=== Semgrep Findings by File ===')
print(df_semgrep.groupby('file').size(), '\n')

## 5. Combined Vulnerability Table

In [ ]:
combined = []

for r in bandit_findings:
    combined.append({
        'tool': 'Bandit',
        'severity': r['severity'],
        'rule': r['rule'],
        'cwe': r['cwe'],
        'file': r['file'],
        'line': r['line'],
        'message': r['text']
    })

for r in semgrep_findings:
    combined.append({
        'tool': 'Semgrep',
        'severity': 'N/A',
        'rule': r['rule'],
        'cwe': 'N/A',
        'file': r['file'],
        'line': r['line'],
        'message': r['message']
    })

df_combined = pd.DataFrame(combined)
df_combined.head(20)

### Auto‑Number Findings (F‑01 → F‑XX)

In [ ]:
df_combined = df_combined.sort_values(by=['file', 'line']).reset_index(drop=True)
df_combined['Finding_ID'] = ['F-' + str(i+1).zfill(2) for i in range(len(df_combined))]
df_combined[['Finding_ID','severity','rule','cwe','file','line','message']].head(20)

## 6. Final Audit Summary

In [ ]:
print('=== FINAL AUDIT SUMMARY ===')
print(f"Total Findings: {len(df_combined)}")
print(f"Bandit Findings: {len(bandit_findings)}")
print(f"Semgrep Findings: {len(semgrep_findings)}")

print('\nSeverity Breakdown:')
print(df_bandit['severity'].value_counts())

print('\nTop Vulnerability Types:')
print(df_combined['rule'].value_counts().head(10))

## 7. Export Markdown Report

In [ ]:
with open('security-reports/audit_summary.md', 'w') as f:
    f.write(df_combined.to_markdown())
print('Markdown report saved.')